In [ ]:
pip install --upgrade torch torchvision torchaudio




# LLM Attention Visualizer



In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from transformers import BertModel, BertTokenizer, BertForSequenceClassification
from datasets import load_dataset
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
import spacy
import warnings
warnings.filterwarnings('ignore')

In [ ]:
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
device

# Model-1: Base BERT — for attention extraction, taxonomy, rollout, probing


In [ ]:
MODEL_NAME = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
model = BertModel.from_pretrained(MODEL_NAME, output_attentions=True)
model.eval()
model.to(device)



In [ ]:
model.config.num_hidden_layers

In [ ]:
print(f"heads-{model.config.num_attention_heads}")
print(f"hidden-{model.config.hidden_size}")


# Model 2: Fine-tuned BERT — ONLY for ablation study


In [ ]:
CLF_MODEL_NAME = "textattack/bert-base-uncased-SST-2"
clf_model = BertForSequenceClassification.from_pretrained(CLF_MODEL_NAME)
clf_model.eval()
clf_model.to(device)


In [ ]:
print(f"num labels-{clf_model.config.num_labels}")

In [ ]:
def get_attention_data(sentence: str):

    encoded = tokenizer(
        sentence,
        return_tensors='pt',
        add_special_tokens=True
    )
    input_ids      = encoded['input_ids'].to(device)
    attention_mask = encoded['attention_mask'].to(device)

    tokens = tokenizer.convert_ids_to_tokens(input_ids[0].tolist())

    with torch.no_grad():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_attentions=True,
            output_hidden_states=True
        )

    # outputs.attentions-tuple of 12 tensors, each [1, 12, seq, seq]
    attention = torch.stack(outputs.attentions, dim=0)  # [12, 1, 12, seq, seq]
    attention = attention.squeeze(1).cpu().numpy()       # [12, 12, seq, seq]

    # outputs.hidden_states-tuple of 13 tensors, each [1, seq, 768]
    hidden_states = torch.stack(outputs.hidden_states, dim=0)  # [13, 1, seq, 768]
    hidden_states = hidden_states.squeeze(1).cpu().numpy()      # [13, seq, 768]

    return tokens, attention, hidden_states




In [ ]:
test_sent = "The cat sat on the mat near the window."
tokens, attention, hidden_states = get_attention_data(test_sent)



In [ ]:
tokens, len(tokens)

In [ ]:
attention.shape

In [ ]:
hidden_states.shape

In [ ]:
attention[0, 0, 1, :].sum()


In [ ]:
def plot_attention_heatmap(tokens, attention, layer: int, head: int):

    attn_matrix = attention[layer, head]  # [seq, seq]

    fig = go.Figure(data=go.Heatmap(
        z=attn_matrix,
        x=tokens,
        y=tokens,
        colorscale='Blues',
        text=np.round(attn_matrix, 3),
        hovertemplate=(
            "Query (FROM): <b>%{y}</b><br>"
            "Key   (TO)  : <b>%{x}</b><br>"
            "Attention   : <b>%{z:.4f}</b>"
            "<extra></extra>"
        )
    ))
    fig.update_layout(
        title=dict(text=f"Attention Heatmap — Layer {layer+1}, Head {head+1}",
                   font=dict(size=16)),
        xaxis=dict(title="Key (attending TO)", tickangle=45, side='bottom'),
        yaxis=dict(title="Query (attending FROM)", autorange='reversed'),
        width=600, height=560,
        margin=dict(l=80, r=20, t=60, b=100)
    )
    fig.show()


def plot_all_heads_grid(tokens, attention, layer: int):

    n_heads = attention.shape[1]
    fig = make_subplots(
        rows=3, cols=4,
        subplot_titles=[f"Head {h+1}" for h in range(n_heads)],
        horizontal_spacing=0.05,
        vertical_spacing=0.08
    )
    for h in range(n_heads):
        row = h // 4 + 1
        col = h % 4 + 1
        fig.add_trace(
            go.Heatmap(
                z=attention[layer, h],
                colorscale='Blues',
                showscale=(h == 0),
                hovertemplate=(
                    f"Layer {layer+1}, Head {h+1}<br>"
                    "From: %{y}<br>To: %{x}<br>"
                    "Weight: %{z:.3f}<extra></extra>"
                )
            ),
            row=row, col=col
        )
        if h == 0:
            fig.update_xaxes(
                ticktext=tokens,
                tickvals=list(range(len(tokens))),
                row=row, col=col
            )
            fig.update_yaxes(
                ticktext=tokens,
                tickvals=list(range(len(tokens))),
                autorange='reversed',
                row=row, col=col
            )
    fig.update_layout(
        title=dict(text=f"All 12 Heads — Layer {layer+1}", font=dict(size=16)),
        height=700, width=950
    )
    fig.show()

In [ ]:
# single head - layer 1, head 1
plot_attention_heatmap(tokens, attention, layer=0, head=0)


In [ ]:
# all 12 heads of layer 5
plot_all_heads_grid(tokens, attention, layer=4)


In [ ]:
new_sent = "John said that he was really tired after the long meeting."
coref_tokens, coref_attn, _ = get_attention_data(new_sent)
plot_attention_heatmap(coref_tokens, coref_attn, layer=7, head=3)

In [ ]:
def compute_head_entropy(attn_matrix):
    """
    normalized shannon entropy across all query rows.
    0 menas perfectly focused on one token
    1 means perfectly uniform across all tokens
    """
    eps = 1e-9
    entropy_per_row = -np.sum(attn_matrix * np.log(attn_matrix + eps), axis=-1)
    max_entropy = np.log(attn_matrix.shape[-1])
    return float(np.mean(entropy_per_row / max_entropy))


def compute_vertical_score(attn_matrix):

    cls_col = float(np.mean(attn_matrix[:, 0]))
    sep_col = float(np.mean(attn_matrix[:, -1]))
    return max(cls_col, sep_col)


def compute_diagonal_score(attn_matrix):

    scores = []
    for offset in [-1, 0, 1]:
        diag = np.diagonal(attn_matrix, offset=offset)
        scores.append(float(np.mean(diag)))
    return max(scores)


def classify_head(attn_matrix,
                  entropy_threshold=0.75,
                  vertical_threshold=0.4,
                  diagonal_threshold=0.4):

    entropy  = compute_head_entropy(attn_matrix)
    vertical = compute_vertical_score(attn_matrix)
    diagonal = compute_diagonal_score(attn_matrix)

    if entropy > entropy_threshold:
        label = 'broad'
    elif vertical > vertical_threshold:
        label = 'vertical'
    elif diagonal > diagonal_threshold:
        label = 'positional'
    else:
        label = 'focused'

    return label, entropy, vertical, diagonal


def analyze_all_heads(attention):

    records = []
    for l in range(attention.shape[0]):
        for h in range(attention.shape[1]):
            label, entropy, vertical, diagonal = classify_head(attention[l, h])
            records.append({
                'layer'         : l + 1,
                'head'          : h + 1,
                'type'          : label,
                'entropy'       : round(entropy, 3),
                'vertical_score': round(vertical, 3),
                'diagonal_score': round(diagonal, 3)
            })
    return pd.DataFrame(records)



In [ ]:

def plot_head_taxonomy(df):
    type_to_num = {'broad': 0, 'vertical': 1, 'positional': 2, 'focused': 3}
    n_layers = df['layer'].max()
    n_heads  = df['head'].max()

    grid_num  = np.zeros((n_layers, n_heads))
    grid_abbr = [['' for _ in range(n_heads)] for _ in range(n_layers)]

    for _, row in df.iterrows():
        l = int(row['layer']) - 1
        h = int(row['head'])  - 1
        grid_num[l, h]  = type_to_num[row['type']]
        grid_abbr[l][h] = row['type'][:3].upper()

    cmap = mcolors.ListedColormap(['#B8B4F0', '#8ADAC5', '#F5C871', '#ED9B84'])
    fig, ax = plt.subplots(figsize=(14, 7))
    ax.imshow(grid_num, cmap=cmap, vmin=-0.5, vmax=3.5, aspect='auto')

    for l in range(n_layers):
        for h in range(n_heads):
            row = df[(df['layer'] == l+1) & (df['head'] == h+1)].iloc[0]
            ax.text(h, l,
                    f"{grid_abbr[l][h]}\n{row['entropy']:.2f}",
                    ha='center', va='center',
                    fontsize=7, color='#1a1a1a')

    ax.set_xticks(range(n_heads))
    ax.set_xticklabels([f"H{h+1}" for h in range(n_heads)], fontsize=9)
    ax.set_yticks(range(n_layers))
    ax.set_yticklabels([f"L{l+1}" for l in range(n_layers)], fontsize=9)
    ax.set_xlabel("Head", fontsize=11)
    ax.set_ylabel("Layer", fontsize=11)
    ax.set_title("Head Behavior Taxonomy — All 144 Heads\n"
                 "BRO=broad  VER=vertical  POS=positional  FOC=focused",
                 fontsize=12, pad=12)
    legend_elements = [
        Patch(facecolor='#B8B4F0', label='Broad (diffuse)'),
        Patch(facecolor='#8ADAC5', label='Vertical ([CLS]/[SEP] sink)'),
        Patch(facecolor='#F5C871', label='Positional (adjacent tokens)'),
        Patch(facecolor='#ED9B84', label='Focused (possibly syntactic)'),
    ]
    ax.legend(handles=legend_elements, loc='upper right',
              bbox_to_anchor=(1.28, 1.0), fontsize=9)
    plt.tight_layout()
    plt.show()


def plot_entropy_heatmap(df):
    n_layers = df['layer'].max()
    n_heads  = df['head'].max()
    grid = np.zeros((n_layers, n_heads))
    for _, row in df.iterrows():
        grid[int(row['layer'])-1, int(row['head'])-1] = row['entropy']

    fig = go.Figure(data=go.Heatmap(
        z=grid,
        x=[f"H{h+1}" for h in range(n_heads)],
        y=[f"L{l+1}" for l in range(n_layers)],
        colorscale='RdYlBu_r',
        text=np.round(grid, 2),
        texttemplate="%{text}",
        hovertemplate="Layer %{y}, Head %{x}<br>Entropy: %{z:.3f}<extra></extra>"
    ))
    fig.update_layout(
        title="Attention Entropy — All 144 Heads (0=focused, 1=diffuse)",
        xaxis_title="Head", yaxis_title="Layer",
        height=450, width=750
    )
    fig.show()


In [ ]:
head_df = analyze_all_heads(attention)



In [ ]:
head_df['type'].value_counts()

In [ ]:
head_df.loc[head_df['entropy'].idxmin()] #most focused head or lowest entropy


In [ ]:
head_df.loc[head_df['entropy'].idxmax()] # most diffuse head or highest entropy

In [ ]:
plot_head_taxonomy(head_df)


In [ ]:
plot_entropy_heatmap(head_df)

In [ ]:
def compute_attention_rollout(attention, head_fusion='mean'):
    """
    Attention Rollout - Abnar & Zuidema (2020).
    """
    n_layers, n_heads, seq_len, _ = attention.shape
    identity = np.eye(seq_len)
    rollout  = identity.copy()

    for layer in range(n_layers):
        if head_fusion == 'mean':
            fused = np.mean(attention[layer], axis=0)
        elif head_fusion == 'max':
            fused = np.max(attention[layer], axis=0)
        else:
            raise ValueError(f"head_fusion must be 'mean' or 'max', got {head_fusion}")

        fused = fused + identity
        fused = fused / fused.sum(axis=-1, keepdims=True)
        rollout = fused @ rollout

    return rollout



In [ ]:

def plot_rollout_cls_bar(tokens, rollout):
    cls_row   = rollout[0, :]
    cls_norm  = cls_row / cls_row.sum()
    colors    = px.colors.sample_colorscale('Blues',
                    list(cls_norm / cls_norm.max()))
    fig = go.Figure(data=go.Bar(
        x=tokens, y=cls_norm,
        marker_color=colors,
        hovertemplate="Token: <b>%{x}</b><br>"
                      "Influence on [CLS]: <b>%{y:.4f}</b><extra></extra>"
    ))
    fig.update_layout(
        title="Attention Rollout — Token Influence on [CLS]",
        xaxis_title="Token", yaxis_title="Normalized Influence",
        height=400, width=700
    )
    fig.show()


def plot_rollout_matrix(tokens, rollout):
    fig = go.Figure(data=go.Heatmap(
        z=rollout,
        x=tokens, y=tokens,
        colorscale='Purples',
        hovertemplate=(
            "Final token : <b>%{y}</b><br>"
            "Input token : <b>%{x}</b><br>"
            "Influence   : <b>%{z:.4f}</b><extra></extra>"
        )
    ))
    fig.update_layout(
        title="Full Attention Rollout Matrix — True Information Flow",
        xaxis=dict(title="Input token (source)", tickangle=45),
        yaxis=dict(title="Output token (destination)", autorange='reversed'),
        height=560, width=620,
        margin=dict(l=80, r=20, t=60, b=100)
    )
    fig.show()


def plot_raw_vs_rollout(tokens, attention, rollout, query_idx=0):

    raw  = np.mean(attention[-1], axis=0)[query_idx]
    roll = rollout[query_idx]
    roll = roll / roll.sum()

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=[
            f"Raw attention (last layer avg)\nQuery: '{tokens[query_idx]}'",
            f"Attention rollout (all 12 layers)\nQuery: '{tokens[query_idx]}'"
        ]
    )
    fig.add_trace(
        go.Bar(x=tokens, y=raw,
               marker_color='#85B7EB', name='Raw'),
        row=1, col=1
    )
    fig.add_trace(
        go.Bar(x=tokens, y=roll,
               marker_color='#AFA9EC', name='Rollout'),
        row=1, col=2
    )
    fig.update_layout(
        title="Raw Attention vs Attention Rollout",
        height=380, width=950, showlegend=False
    )
    fig.show()

In [ ]:
rollout = compute_attention_rollout(attention, head_fusion='mean')


In [ ]:
rollout.shape

In [ ]:
rollout[0].sum()  #[CLS] row sums to

In [ ]:
plot_rollout_cls_bar(tokens, rollout)


In [ ]:
plot_rollout_matrix(tokens, rollout)


In [ ]:
plot_raw_vs_rollout(tokens, attention, rollout, query_idx=0)


In [ ]:
# Also run on a sentiment sentence — shows what BERT focuses on
sentiment = "This movie was absolutely terrible and a complete waste of time."
sent_tokens, sent_attn, _ = get_attention_data(sentiment)
sent_rollout = compute_attention_rollout(sent_attn)
plot_rollout_cls_bar(sent_tokens, sent_rollout)

In [ ]:
def predict_batch(model, tokenizer, data, device):
    """
    evaluate model accuracy on list of sentence and label tuples.
    """
    correct = 0
    for sentence, true_label in data:
        inputs = tokenizer(
            sentence,
            return_tensors='pt',
            truncation=True,
            max_length=128
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            logits = model(**inputs).logits
        pred = logits.argmax(dim=-1).item()
        correct += int(pred == true_label)
    return correct / len(data)


def ablate_head(model, layer_idx: int, head_idx: int):

    head_size = model.config.hidden_size // model.config.num_attention_heads
    start = head_idx * head_size
    end   = start + head_size

    attn_module = model.bert.encoder.layer[layer_idx].attention.self

    def hook_fn(module, input, output):
        # output[0] shape: [batch, seq_len, hidden_size]
        output[0][:, :, start:end] = 0.0
        return output

    return attn_module.register_forward_hook(hook_fn)


def run_ablation_study(clf_model, tokenizer, data, device,
                       n_layers=12, n_heads=12):

    baseline = predict_batch(clf_model, tokenizer, data, device)

    print(f"baseline accuracy: {baseline:.4f} ({baseline*100:.1f}%)")
    print(f"running ablation on {n_layers * n_heads} heads...\n")

    importance = np.zeros((n_layers, n_heads))

    for l in range(n_layers):
        for h in range(n_heads):
            handle     = ablate_head(clf_model, l, h)
            ablated    = predict_batch(clf_model, tokenizer, data, device)
            handle.remove()
            importance[l, h] = baseline - ablated

        print(f"  Layer {l+1:2d} done | "
              f"Max drop this layer: {importance[l].max():.4f} | "
              f"Min drop: {importance[l].min():.4f}")

    return baseline, importance


def plot_ablation_heatmap(importance, baseline):

    n_layers, n_heads = importance.shape
    fig = go.Figure(data=go.Heatmap(
        z=importance,
        x=[f"H{h+1}" for h in range(n_heads)],
        y=[f"L{l+1}" for l in range(n_layers)],
        colorscale='RdBu_r',
        zmid=0,
        text=np.round(importance, 3),
        texttemplate="%{text}",
        hovertemplate=(
            "Layer %{y}, Head %{x}<br>"
            "Accuracy drop: <b>%{z:.4f}</b><extra></extra>"
        )
    ))
    fig.update_layout(
        title=(f"Head Ablation — Accuracy Drop Per Head "
               f"(baseline={baseline:.3f})<br>"
               "<sup>Red = critical | Blue = redundant</sup>"),
        xaxis_title="Head", yaxis_title="Layer",
        height=480, width=750
    )
    fig.show()

    # print summary
    threshold = 0.01
    critical  = int(np.sum(importance >  threshold))
    redundant = int(np.sum(importance <= threshold))
    best_l, best_h = np.unravel_index(importance.argmax(), importance.shape)
    print(f"\nAblation Summary (baseline={baseline:.3f}):")
    print(f"  Total heads     : {importance.size}")
    print(f"  Critical (>{threshold*100:.0f}% drop): "
          f"{critical} ({critical/importance.size*100:.0f}%)")
    print(f"  Redundant       : "
          f"{redundant} ({redundant/importance.size*100:.0f}%)")
    print(f"  Most critical   : Layer {best_l+1}, Head {best_h+1} "
          f"(drop = {importance[best_l, best_h]:.4f})")
    print(f"  Mean drop       : {importance.mean():.4f}")

In [ ]:
# load SST-2 validation data
sst2 = load_dataset('nyu-mll/glue', 'sst2', split='validation')

In [ ]:
eval_data = [(ex['sentence'], ex['label']) for ex in sst2.select(range(872))]


In [ ]:
eval_data

In [ ]:
len(eval_data)

In [ ]:
eval_data[0][0], eval_data[0][1]

In [ ]:
baseline, importance = run_ablation_study(
    clf_model, tokenizer, eval_data, device
)



In [ ]:
plot_ablation_heatmap(importance, baseline)

In [ ]:
def run_layerwise_ablation(clf_model, tokenizer, data, device):
    """
    now remove all heads from one layer
    """
    baseline = predict_batch(clf_model, tokenizer, data, device)
    print(f"baseline-{baseline:.4f}")

    layer_importance = []

    for l in range(12):
        # Ablate all 12 heads in this layer simultaneously
        handles = [ablate_head(clf_model, l, h) for h in range(12)]
        acc = predict_batch(clf_model, tokenizer, data, device)
        for h in handles:
            h.remove()

        drop = baseline - acc
        layer_importance.append(drop)
        print(f"layer {l+1:2d} | accuracy: {acc:.4f} | drop: {drop:.4f}")

    # Plot
    fig = go.Figure(data=go.Bar(
        x=[f"L{l+1}" for l in range(12)],
        y=layer_importance,
        marker_color=['#ED9B84' if d > 0.01 else '#8ADAC5'
                      for d in layer_importance],
        hovertemplate="Layer %{x}<br>Accuracy drop: <b>%{y:.4f}</b><extra></extra>"
    ))
    fig.update_layout(
        title=f"Layer-wise Ablation — Full Layer Removal Impact (baseline={baseline:.3f})",
        xaxis_title="Layer", yaxis_title="Accuracy Drop",
        height=400, width=700
    )
    fig.show()
    return layer_importance

layer_drops = run_layerwise_ablation(clf_model, tokenizer, eval_data, device)

1% threshold is just arb

In [ ]:
layer_drops_array = np.array(layer_drops)
layers = [f"L{l+1}" for l in range(12)]

# Color code: red = critical, yellow = moderate, green = redundant
colors = []
for d in layer_drops:
    if d >= 0.010:
        colors.append('#E05C5C')   # red — critical
    elif d >= 0.005:
        colors.append('#F5C871')   # yellow — moderate
    else:
        colors.append('#8ADAC5')   # green — redundant

fig = go.Figure()

# Bar chart
fig.add_trace(go.Bar(
    x=layers,
    y=layer_drops_array,
    marker_color=colors,
    hovertemplate="Layer %{x}<br>Drop: <b>%{y:.4f}</b><extra></extra>",
    name='Accuracy Drop'
))

# Threshold line at 1%( this is just arb)
fig.add_hline(
    y=0.01,
    line_dash='dash',
    line_color='red',
    annotation_text='1% threshold',
    annotation_position='right'
)

fig.update_layout(
    title=(f"Layer-wise Ablation — Full Layer Removal<br>"
           f"<sup>baseline={0.9243:.3f} | "
           f"Most critical: Layer 7 & 9 (drop=1.15%)</sup>"),
    xaxis_title="Layer",
    yaxis_title="Accuracy Drop (higher = more important)",
    height=450, width=750
)
fig.show()

# Print clean summary
print("\nLayer Importance Ranking:")
ranked = sorted(enumerate(layer_drops), key=lambda x: x[1], reverse=True)
for rank, (l_idx, drop) in enumerate(ranked):
    bar   = '█' * int(drop * 3000)
    label = '← CRITICAL' if drop >= 0.010 else ''
    print(f"  #{rank+1:2d}  Layer {l_idx+1:2d} | {drop:.4f} | {bar} {label}")

now based on σ

In [ ]:
layer_drops_array = np.array(layer_drops)
layers = [f"L{l+1}" for l in range(12)]

mean_drop = layer_drops_array.mean()
std_drop  = layer_drops_array.std()

# σ-based zones:
#   > mean + 1.5σ  → critical
#   > mean + 0.5σ  → notable
#   everything else → redundant
critical_thresh = mean_drop + 1.5 * std_drop
notable_thresh  = mean_drop + 0.5 * std_drop

print(f"Mean drop : {mean_drop:.4f}")
print(f"Std dev   : {std_drop:.4f}")
print(f"Critical  : drop > {critical_thresh:.4f} (mean + 1.5σ)")
print(f"Notable   : drop > {notable_thresh:.4f}  (mean + 0.5σ)")
print()

colors = []
labels = []
for d in layer_drops:
    if d > critical_thresh:
        colors.append('#E05C5C')
        labels.append('critical')
    elif d > notable_thresh:
        colors.append('#F5C871')
        labels.append('notable')
    else:
        colors.append('#8ADAC5')
        labels.append('redundant')

fig = go.Figure()

# Bars
fig.add_trace(go.Bar(
    x=layers,
    y=layer_drops_array,
    marker_color=colors,
    hovertemplate=(
        "Layer %{x}<br>"
        "Drop: <b>%{y:.4f}</b><br>"
        "Baseline drop: " + f"{mean_drop:.4f}" +
        "<extra></extra>"
    ),
    name='Accuracy Drop'
))

# Mean line
fig.add_hline(
    y=mean_drop,
    line_dash='dot',
    line_color='#94A3B8',
    annotation_text=f'mean ({mean_drop:.4f})',
    annotation_position='left',
    annotation_font_color='#94A3B8'
)

# mean + 0.5σ line
fig.add_hline(
    y=notable_thresh,
    line_dash='dash',
    line_color='#F5C871',
    annotation_text=f'mean + 0.5σ ({notable_thresh:.4f})',
    annotation_position='right',
    annotation_font_color='#F5C871'
)

# mean + 1.5σ line
fig.add_hline(
    y=critical_thresh,
    line_dash='dash',
    line_color='#E05C5C',
    annotation_text=f'mean + 1.5σ ({critical_thresh:.4f})',
    annotation_position='right',
    annotation_font_color='#E05C5C'
)

fig.update_layout(
    title=(
        f"Layer-wise Ablation — σ-Based Importance Classification<br>"
        f"<sup>baseline={0.9243:.3f} | "
        f"mean={mean_drop:.4f} | "
        f"σ={std_drop:.4f} | "
        f"critical = mean + 1.5σ</sup>"
    ),
    xaxis_title="Layer",
    yaxis_title="Accuracy Drop (higher = more important)",
    height=500,
    width=800,
    plot_bgcolor='#0A0E1A',
    paper_bgcolor='#0A0E1A',
    font_color='#F1F5F9',
    xaxis=dict(gridcolor='#1E293B'),
    yaxis=dict(gridcolor='#1E293B'),
)
fig.show()

print(f"{'Rank':<6} {'Layer':<8} {'Drop':<10} "
      f"{'σ distance':<14} {'Zone':<12} {'Bar'}")
print("─" * 65)

ranked = sorted(enumerate(layer_drops),
                key=lambda x: x[1], reverse=True)

for rank, (l_idx, drop) in enumerate(ranked):
    sigma_dist = (drop - mean_drop) / std_drop
    zone       = labels[l_idx]
    bar_len    = int(drop * 3000)
    bar        = '█' * bar_len

    # color label for terminal
    zone_display = {
        'critical' : 'critical',
        'notable'  : 'notable',
        'redundant': 'redundant'
    }[zone]

    print(f"  #{rank+1:<4} L{l_idx+1:<6} "
          f"{drop:.4f}     "
          f"{sigma_dist:+.2f}σ         "
          f"{zone_display:<20} {bar}")


critical_layers  = [i+1 for i, l in enumerate(labels)
                    if l == 'critical']
notable_layers   = [i+1 for i, l in enumerate(labels)
                    if l == 'notable']
redundant_layers = [i+1 for i, l in enumerate(labels)
                    if l == 'redundant']

print(f"Critical layers  : {critical_layers}")
print(f"Notable layers   : {notable_layers}")
print(f"Redundant layers : {redundant_layers}")


In [ ]:
nlp_spacy = spacy.load("en_core_web_sm")


In [ ]:
def get_bert_pos_alignment(sentence: str, tokens: list):

    doc = nlp_spacy(sentence)
    spacy_tokens = [(t.text, t.pos_) for t in doc]

    pos_labels = [None]  # [CLS]
    spacy_idx  = 0

    for bert_tok in tokens[1:-1]:  # skip [CLS] and [SEP]
        if bert_tok.startswith("##"):
            # subword — inherit POS from current word
            pos_labels.append(spacy_tokens[spacy_idx-1][1]
                               if spacy_idx > 0 else "X")
        else:
            # new word — find matching spacy token
            if spacy_idx < len(spacy_tokens):
                pos_labels.append(spacy_tokens[spacy_idx][1])
                spacy_idx += 1
            else:
                pos_labels.append("X")

    pos_labels.append(None)  # [SEP]
    return pos_labels


def probe_all_heads_for_pos(sentences: list, attention_list: list,
                             tokens_list: list):

    # Step 1: find max sequence length across all sentences
    max_seq_len = max(a.shape[2] for a in attention_list)

    all_features = {(l, h): [] for l in range(12) for h in range(12)}
    all_labels   = []

    for sentence, tokens, attention in zip(sentences, tokens_list,
                                           attention_list):
        pos_labels = get_bert_pos_alignment(sentence, tokens)
        seq_len    = attention.shape[2]

        for i, (token, pos) in enumerate(zip(tokens, pos_labels)):
            if pos is None:
                continue  # skip [CLS] and [SEP]

            all_labels.append(pos)

            for l in range(12):
                for h in range(12):
                    row = attention[l, h, i, :]  # shape: [seq_len]

                    # Pad to max_seq_len with zeros so all rows same size
                    padded = np.zeros(max_seq_len)
                    padded[:seq_len] = row

                    all_features[(l, h)].append(padded)

    le = LabelEncoder()
    y  = le.fit_transform(all_labels)

    print(f"total tokens collected : {len(all_labels)}")
    print(f"pos classes            : {list(le.classes_)}")
    print(f"feature size per token : {max_seq_len}")

    records = []

    for l in range(12):
        for h in range(12):
            X = np.array(all_features[(l, h)])   #[n_tokens, max_seq_len]
            if len(X) < 10:
                continue
            clf = LogisticRegression(max_iter=500, random_state=42, C=1.0)
            clf.fit(X, y)
            acc = accuracy_score(y, clf.predict(X))
            records.append({
                'layer'       : l + 1,
                'head'        : h + 1,
                'pos_accuracy': round(acc, 3)
            })
        print(f"layer {l+1:2d} probed")

    return pd.DataFrame(records)


def plot_probing_heatmap(probe_df):
    n_layers = probe_df['layer'].max()
    n_heads  = probe_df['head'].max()
    grid     = np.zeros((n_layers, n_heads))

    for _, row in probe_df.iterrows():
        grid[int(row['layer'])-1, int(row['head'])-1] = row['pos_accuracy']

    fig = go.Figure(data=go.Heatmap(
        z=grid,
        x=[f"H{h+1}" for h in range(n_heads)],
        y=[f"L{l+1}" for l in range(n_layers)],
        colorscale='Greens',
        text=np.round(grid, 2),
        texttemplate="%{text}",
        hovertemplate=(
            "Layer %{y}, Head %{x}<br>"
            "POS Accuracy: <b>%{z:.3f}</b><extra></extra>"
        )
    ))
    fig.update_layout(
        title=("POS Probing Accuracy — Which Heads Encode Grammar?<br>"
               "<sup>Higher = this head's attention pattern predicts "
               "part-of-speech tags</sup>"),
        xaxis_title="Head", yaxis_title="Layer",
        height=450, width=750
    )
    fig.show()

    best = probe_df.loc[probe_df['pos_accuracy'].idxmax()]
    print(f"\nbest POS head: Layer {int(best['layer'])}, "
          f"Head {int(best['head'])} "
          f"(accuracy = {best['pos_accuracy']:.3f})")


In [ ]:
probe_sent = [
    "The cat sat on the mat near the window.",
    "John said that he was really tired after the long meeting.",
    "The company released its quarterly earnings report yesterday.",
    "Scientists discovered a new species of bird in the Amazon rainforest.",
    "The government announced new policies to address climate change.",
    "She quickly ran to catch the last train to the city center.",
    "The old library contains thousands of rare and valuable books.",
    "His ambitious plan to reform the education system failed completely.",
    "The young engineer designed an innovative solution to the problem.",
    "After years of research, the team finally published their findings.",
]


In [ ]:
probe_tokens_list  = []
probe_attn_list    = []

In [ ]:
for sent in probe_sent:
    t, a, _ = get_attention_data(sent)
    probe_tokens_list.append(t)
    probe_attn_list.append(a)

In [ ]:
probe_df = probe_all_heads_for_pos(
    probe_sent, probe_attn_list, probe_tokens_list
)


In [ ]:
plot_probing_heatmap(probe_df)